In [0]:
display(dbutils.fs.ls("/databricks-datasets/nyctaxi/tripdata/yellow/"))

In [0]:
df_raw = spark.read.parquet("/databricks-datasets/nyctaxi/tripdata/yellow/")
display(df_raw.limit(10))

In [0]:
files = dbutils.fs.ls("/databricks-datasets/nyctaxi/tripdata/yellow/")
for f in files:
    print(f.name)

In [0]:
# Leer solo 1 archivo para ver el esquema
df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/databricks-datasets/nyctaxi/tripdata/yellow/yellow_tripdata_2019-01.csv.gz")

print(df_raw.columns)
display(df_raw.limit(5))

In [0]:
from pyspark.sql.functions import col

# Leer todos los meses de 2019
df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/databricks-datasets/nyctaxi/tripdata/yellow/yellow_tripdata_2019-*.csv.gz")

# Limpiar y filtrar
df_silver = df_raw.filter(
    (col("trip_distance") > 0) &
    (col("total_amount") > 0) &
    (col("passenger_count").between(1, 6)) &
    (col("tpep_pickup_datetime").isNotNull()) &
    (col("tpep_dropoff_datetime").isNotNull()) &
    (col("fare_amount") > 0)
).select(
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "RatecodeID",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "fare_amount",
    "tip_amount",
    "tolls_amount",
    "total_amount",
    "congestion_surcharge"
)

# Guardar en Silver
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("proyecto_bi.silver.yellow_trips")

print("✅ Tabla Silver creada exitosamente")

In [0]:
%sql
SELECT COUNT(*) FROM proyecto_bi.silver.yellow_trips